# Rubric-Based On-Policy Distillation (ROPD) — Qwen3-0.6B

Instead of minimising token-level KL against a teacher, we score each student rollout
with a **rubric** — checkable criteria for format and factual correctness — and train
with **GRPO** (Group Relative Policy Optimization).

```
question
   │
   ▼
STUDENT (no manual) ──► G rollouts
                              │
                         rubric score each
                         ┌──────────────────────────┐
                         │ format:   </think> present?  +1.0 / -1.0  │
                         │           answer non-empty?  +0.5         │
                         │           think block short? +0.3         │
                         │ factual:  ref keywords found? 0..+1.0     │
                         └──────────────────────────┘
                              │
                    advantage = reward - mean(group rewards)
                              │
                    REINFORCE: loss = -advantage * log_prob(rollout)
                              │
                    backprop into student LoRA
```

**No teacher forward pass.** Rubric is pure regex + keyword matching.

**Runtime:** Colab → Runtime → Change runtime type → **L4 GPU**  
**Prerequisites:** run `sft_qwen3.ipynb` first to produce `qwen3_06b_ac_sft_lora/`.
Upload the adapter folder, `ac_manual_synth_tra.jsonl`, and `ac_manual_synth_val.jsonl`.


In [ ]:
# 0. Install. After this runs: Runtime → Restart session, then Run all.
!pip install -q "transformers>=4.51" "peft>=0.13" accelerate


In [ ]:
# 1. Config
import re, random, json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID   = "Qwen/Qwen3-0.6B"
ADAPTER_DIR = "./qwen3_06b_ac_sft_lora"   # SFT adapter from sft_qwen3.ipynb
TRAIN_JSONL = "ac_manual_synth_tra.jsonl"
VAL_JSONL   = "ac_manual_synth_val.jsonl"
OUTPUT_DIR  = "./qwen3_06b_ac_ropd_lora"

SYSTEM = ("You are a helpful assistant for the Sharp CV-P09FX portable air conditioner. "
          "Answer questions accurately based on the product manual.")

# GRPO knobs
G              = 4      # rollouts per question; more = lower variance, more compute
EPOCHS         = 3
N_TRAIN_QS     = 60     # fact-dense subset of training questions
MAX_NEW_TOKENS = 384    # generous budget; rubric penalises long outputs anyway
GEN_TEMP       = 0.9    # sample diverse rollouts within a group
GEN_TOP_P      = 0.95
LR             = 5e-5   # lower than SFT; GRPO gradient is noisier
ACCUM_STEPS    = 4
SEED           = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32
random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE, "| dtype:", DTYPE)


In [ ]:
# 2. Data loading
_USER_RE = re.compile(r"<\|im_start\|>user\n(.*?)<\|im_end\|>", re.DOTALL)
_ASST_RE = re.compile(r"<\|im_start\|>assistant\n(.*?)<\|im_end\|>", re.DOTALL)

_STOPWORDS = {"that", "this", "with", "from", "have", "will", "when", "your",
              "must", "should", "about", "after", "before", "which", "their",
              "there", "where", "these", "those", "would", "could", "using"}

def extract_keywords(answer_text):
    """Pull numbers and technical terms from a reference answer.
    Used to build per-question factual rubric criteria."""
    # Numbers (including decimals and unit-attached like 559mm)
    numbers = re.findall(r'\b\d+(?:\.\d+)?\b', answer_text)
    # Technical words (5+ chars, not stopwords)
    words = re.findall(r'\b[a-zA-Z]{5,}\b', answer_text.lower())
    words = [w for w in words if w not in _STOPWORDS]
    # Deduplicate, keep up to 8 keywords total
    seen = set()
    keywords = []
    for kw in numbers + words:
        if kw not in seen:
            seen.add(kw)
            keywords.append(kw)
        if len(keywords) >= 8:
            break
    return keywords

def load_qa_pairs(path):
    """Load JSONL; return list of (question, final_answer, keywords)."""
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            text = json.loads(line)["text"]
            qm = _USER_RE.search(text)
            am = _ASST_RE.search(text)
            if not (qm and am):
                continue
            q = qm.group(1).strip()
            a_full = am.group(1).strip()
            # Strip think block — rubric keywords come from the final answer only
            answer = a_full.split("</think>", 1)[1].strip() if "</think>" in a_full else a_full
            pairs.append((q, answer, extract_keywords(answer)))
    return pairs

train_pairs = load_qa_pairs(TRAIN_JSONL)
val_pairs   = load_qa_pairs(VAL_JSONL)
print(f"Train: {len(train_pairs)} pairs | Val: {len(val_pairs)} pairs")

# Fact-dense subset: prioritise questions whose reference answer has many numbers
_FACT_KW = ["22", "559", "inch", "mm", "screw", "drain", "install",
             "width", "window", "exhaust", "filter", "reset", "power", "panel"]
def _is_fact_dense(q):
    return any(kw in q.lower() for kw in _FACT_KW)

fact_first  = [p for p in train_pairs if _is_fact_dense(p[0])]
remaining   = [p for p in train_pairs if not _is_fact_dense(p[0])]
distill_pairs = (fact_first + remaining)[:N_TRAIN_QS]
val_qs = [q for q, _, _ in val_pairs[:3]]   # for before/after eval

print(f"Distillation pairs: {len(distill_pairs)}")
print(f"\nSample keywords:")
for q, a, kws in distill_pairs[:3]:
    print(f"  Q: {q[:60]}")
    print(f"  keywords: {kws}\n")


In [ ]:
# 3. Tokenizer + model (load SFT adapter as starting point)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Resolve </think> token ID at runtime
THINK_END_ID = tokenizer.convert_tokens_to_ids("</think>")
if THINK_END_ID == tokenizer.unk_token_id:
    _ids = tokenizer.encode("</think>", add_special_tokens=False)
    THINK_END_ID = _ids[0] if _ids else None
print(f"</think> token id: {THINK_END_ID}")

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=True)
model.print_trainable_parameters()


In [ ]:
# 4. Prompt builder + generation helpers
def build_prompt_ids(question):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user",   "content": question}]
    ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=True,
        return_tensors="pt", return_dict=True,
    )["input_ids"]
    return ids.to(DEVICE)

def logits_on_gen(model, prompt_ids, gen_ids):
    """Forward pass on [prompt ++ gen]; return logits for each gen token."""
    full   = torch.cat([prompt_ids, gen_ids], dim=1)
    logits = model(full).logits[0].float()
    start  = prompt_ids.shape[1] - 1
    return logits[start : start + gen_ids.shape[1]]

@torch.no_grad()
def sample_rollouts(question):
    """Sample G diverse rollouts from the student. Returns list of (prompt, gen)."""
    model.eval()
    prompt = build_prompt_ids(question)
    rollouts = []
    for _ in range(G):
        out = model.generate(
            prompt, attention_mask=torch.ones_like(prompt),
            max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
            temperature=GEN_TEMP, top_p=GEN_TOP_P,
            pad_token_id=tokenizer.pad_token_id,
        )
        rollouts.append((prompt, out[:, prompt.shape[1]:]))
    return rollouts

@torch.no_grad()
def answer(question):
    """Greedy eval answer with repetition penalty."""
    model.eval()
    prompt = build_prompt_ids(question)
    out = model.generate(
        prompt, attention_mask=torch.ones_like(prompt),
        max_new_tokens=512, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.15,
    )
    return tokenizer.decode(out[0, prompt.shape[1]:], skip_special_tokens=True).strip()


In [ ]:
# 5. Rubric
#
# Two components scored independently:
#   format_score  — is the output well-structured? (always applicable)
#   factual_score — does it mention the right facts? (question-specific keywords)
#
# GRPO only needs relative ranking within a group, so absolute scale doesn't matter much.

def format_score(gen_ids):
    """Score based on output structure.
    +1.0  </think> present (output terminates cleanly)
    -1.0  no </think>  (output is a loop)
    +0.5  non-empty answer after </think>
    -0.3  think block > 150 tokens (probably looped before hitting </think>)
    """
    ids = gen_ids[0].tolist()
    if THINK_END_ID not in ids:
        return -1.0

    think_end_pos = ids.index(THINK_END_ID)
    answer_len    = len(ids) - think_end_pos - 1

    score = 1.0
    if answer_len < 3:
        score -= 0.5   # </think> present but no answer followed
    if think_end_pos > 150:
        score -= 0.3   # very long think block
    return score

def factual_score(gen_ids, ref_keywords):
    """Score based on how many reference keywords appear in the decoded output.
    Returns 0..1 (fraction of keywords found)."""
    if not ref_keywords:
        return 0.0
    text = tokenizer.decode(gen_ids[0], skip_special_tokens=True).lower()
    found = sum(1 for kw in ref_keywords if kw in text)
    return found / len(ref_keywords)

def rubric_score(gen_ids, ref_keywords):
    return format_score(gen_ids) + factual_score(gen_ids, ref_keywords)

# Quick sanity check
print("Rubric OK — components: format_score + factual_score")
print(f"Score range approx: -1.0 (loop, no facts) to +2.0 (clean format + all facts)")


In [ ]:
# 6. Before training — baseline answers
print("=== Before ROPD ===")
for q in val_qs:
    print(f"\nQ: {q}\nA: {answer(q)}")


## Train

Each step:
1. Sample G rollouts from student for one question
2. Score each with the rubric
3. Compute advantages = reward − mean(group rewards)
4. REINFORCE: loss = −advantage × mean_token(log_prob)
5. Skip if all rollouts scored the same (no gradient signal)

Watch the mean reward rise and the `terminated%` (fraction of rollouts that
reached `</think>`) approach 100%. If `terminated%` is already high from the
SFT warmstart, the format reward will be low-variance and factual score will
drive most of the learning.


In [ ]:
opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR
)

step = 0
for epoch in range(EPOCHS):
    random.shuffle(distill_pairs)
    opt.zero_grad()
    epoch_rewards = []; epoch_terminated = 0; epoch_total = 0

    for idx, (question, _, ref_keywords) in enumerate(distill_pairs):

        # --- Phase 1: sample G rollouts (no grad) ---
        rollouts = sample_rollouts(question)
        rewards  = [rubric_score(gen, ref_keywords) for _, gen in rollouts]
        baseline = sum(rewards) / G
        advantages = [r - baseline for r in rewards]

        epoch_rewards.extend(rewards)
        epoch_terminated += sum(
            1 for _, gen in rollouts
            if THINK_END_ID is not None and THINK_END_ID in gen[0].tolist()
        )
        epoch_total += G

        # Skip if zero variance (all rollouts identical reward → no useful gradient)
        if max(advantages) == min(advantages) == 0:
            continue

        # --- Phase 2: policy gradient with grad ---
        model.train()
        count = 0
        for (prompt, gen), adv in zip(rollouts, advantages):
            if gen.shape[1] == 0 or adv == 0:
                continue
            logits   = logits_on_gen(model, prompt, gen)
            log_prob = F.log_softmax(logits, dim=-1)
            token_lp = log_prob.gather(-1, gen.squeeze(0).unsqueeze(-1)).squeeze(-1)
            loss = (-adv * token_lp.mean()) / (G * ACCUM_STEPS)
            loss.backward()
            count += 1

        if count > 0 and (idx + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            opt.step(); opt.zero_grad(); step += 1

            if step % 5 == 0:
                mean_r   = sum(epoch_rewards[-G*ACCUM_STEPS:]) / max(G*ACCUM_STEPS, 1)
                term_pct = 100 * epoch_terminated / max(epoch_total, 1)
                print(f"epoch {epoch}  step {step}  "
                      f"mean_reward={mean_r:.3f}  terminated={term_pct:.0f}%")

    opt.step(); opt.zero_grad()  # flush remainder
    mean_r   = sum(epoch_rewards) / max(len(epoch_rewards), 1)
    term_pct = 100 * epoch_terminated / max(epoch_total, 1)
    print(f"--- epoch {epoch} done  mean_reward={mean_r:.3f}  terminated={term_pct:.0f}% ---\n")

print("Training complete.")


In [ ]:
# 7. After training — compare with baseline
print("=== After ROPD ===")
for q in val_qs:
    print(f"\nQ: {q}\nA: {answer(q)}")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nSaved adapter to {OUTPUT_DIR}")
